# Set up the environment

In [ ]:
%pip install -q --upgrade google-genai



In [ ]:
from IPython.display import HTML, display

def set_css(info):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
%load_ext autoreload
%autoreload 2
%aimport
%load_ext bigquery_magics
import os
import sys
import logging

module_path = os.path.abspath(os.path.join('..'))
sys.path.append(module_path)

format_string = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
logger = logging.getLogger()
fhandler = logging.FileHandler(filename='mylog.log', mode='a')
formatter = logging.Formatter(format_string)
fhandler.setFormatter(formatter)
#logger.addHandler(fhandler)
logging.basicConfig(format=format_string,
                     level=logging.INFO, stream=sys.stdout)
logger.setLevel(logging.INFO)

In [ ]:
PROJECT_ID = "uk-bh-experiments-argolis"  # @param {type:"string"}
REGION = "us-central1"  # @param {type: "string"}
DATASET_NAME = "breedr"  # @param {type: "string"}
SQL_RETRY_TIMES = "3"  # @param {type: "integer


#uncomment if required
#from google.colab import auth
#auth.authenticate_user(project_id=PROJECT_ID)

## Helper Functions

In [ ]:
import re
import json

def format_json(json_dict):
  return f'<div style="text-align:left"><pre>{json.dumps(json_dict, indent=2)}</pre></div>'

def parse_json_markdown(json_string: str) -> dict:
    # Try to find JSON string within first and last triple backticks
    match = re.search(r"""```       # match first occuring triple backticks
                          (?:json)? # zero or one match of string json in non-capturing group
                          (.*)```   # greedy match to last triple backticks""", json_string, flags=re.DOTALL|re.VERBOSE|re.IGNORECASE)

    # If no match found, assume the entire string is a JSON string
    if match is None:
        json_str = json_string
    else:
        # If match found, use the content within the backticks
        json_str = match.group(1)

    # Strip whitespace and newlines from the start and end
    json_str = json_str.strip()

    # Parse the JSON string into a Python dictionary while allowing control characters by setting strict to False
    parsed = json.loads(json_str, strict=False)

    return parsed

def strip_triple_ticks(text: str) -> str:
    """
    Strip a triple tick mark off the start and end of a string if they are present.

    Args:
        text: The string to strip.

    Returns:
        The stripped string.
    """

    if (text.startswith("```") and text.endswith("```")) or (text.startswith("'''") and text.endswith("'''")):
        return text[3:-3]
    else:
        return text

def strip_sql_markdown(text: str) -> str:
  text = text.replace("sql", "")
  text = text.replace("```", "")
  return text

## Configure the AI enviroment

In [ ]:
#import vertexai
#from vertexai.generative_models import GenerativeModel, Part
from google import genai
from google.genai import types


GENAI_MODEL_NAME = "gemini-2.0-flash-001"  # @param {type: "string"}
DEFAULT_TEMP = 1
DEFAULT_TOP_P = 0.95
DEFAULT_MAX_OUTPUT_TOKENS=8192

DEFAULT_SAFETY_SETTINGS = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="OFF"
    )]



In [ ]:
from ask_data.ask_data_core import AskData

ask_data = AskData(model_name=GENAI_MODEL_NAME,project_id=PROJECT_ID, region=REGION)

### Function to call GenAI

In [ ]:
from google import genai
from google.genai import types
from google.genai.types import Part
import base64

def generate(text="", parts=[], temperature=DEFAULT_TEMP, top_p=DEFAULT_TOP_P, model=GENAI_MODEL_NAME, project=PROJECT_ID, location=REGION):

  logger.debug(f'{text=}')
  client = genai.Client(
      vertexai=True,
      project=project,
      location=location,
  )
  if len(parts) == 0:
    parts = [types.Part.from_text(text=text)]
    

  contents = [
    types.Content(
      role="user",
      parts=parts
    )
  ]
  generate_content_config = types.GenerateContentConfig(
    temperature = temperature,
    top_p = top_p,
    max_output_tokens = DEFAULT_MAX_OUTPUT_TOKENS,
    response_modalities = ["TEXT"],
    safety_settings = DEFAULT_SAFETY_SETTINGS
  )

  response = client.models.generate_content(
    model = model,
    contents = contents,
    config = generate_content_config,
    )
  return response  

In [ ]:
response = generate(text="what time is it")
print(f'{response.text=}')

# Understand the Database

## Get Schema

In [ ]:
%%bigquery table_schema
SELECT
  table_name,
  ARRAY_AGG(STRUCT(
    CONCAT(table_name, ".",column_name) as column_name,
    data_type,
    IF(is_nullable = 'YES', 'NULLABLE', 'REQUIRED') AS mode)
  ORDER BY ordinal_position) AS schema
FROM
  uk-bh-experiments-argolis.breedr.INFORMATION_SCHEMA.COLUMNS
  group by table_name

In [ ]:
table_schema.head()

table_names = table_schema.table_name.unique()
table_names

In [ ]:
schema_json = table_schema.to_json(orient='records')
print(schema_json)


##Sample the tables

In [ ]:
import pandas as pd
from google.cloud import bigquery

def get_table_sample(table_name: str, sample_size: int =10) -> pd.DataFrame:
  client = bigquery.Client()

  query = f"""
    SELECT *
    FROM {table_name} TABLESAMPLE SYSTEM (10 PERCENT)
    WHERE rand() < 0.5
    limit {sample_size}
  """
  result = client.query(query).to_dataframe()
  if result.empty:
    print(f"No random sample rows found in {table_name}, getting first row instead")
    query = f"""
      SELECT *
      FROM {table_name}
      limit 1
    """
    result = client.query(query).to_dataframe()

  print(f"Got {result.shape[0]} rows from {table_name}")
  return result

def sample_tables(project_id=PROJECT_ID, dataset_name=DATASET_NAME) -> dict[str, pd.DataFrame]:
  table_samples = {}
  for table_name in table_names:
    print(f"Sampling {table_name}")
    full_table_name = f"{project_id}.{dataset_name}.{table_name}"
    table_samples[table_name] = get_table_sample(full_table_name, 10)
  return table_samples

def get_table_samples_json():
  table_samples = sample_tables()

  table_samples_json = {}
  for table_name in table_names:
    table_samples_json[table_name] = table_samples[table_name].to_json(orient='records')
  return table_samples_json

#table_samples_json = get_table_samples_json()

#table_samples_json

## read the Instructions File

In [ ]:
from google.cloud import storage
from google.cloud.storage import blob

def getFileFromBucket(bucket_name: str, file_name: str)-> blob:
  storage_client = storage.Client()  # Implicit environ set-up
  bucket = storage_client.bucket(bucket_name)
  file_blob = bucket.blob(file_name)
  return file_blob

def getFileFromGSPath(gs_path: str) -> blob:
  bucket_name = gs_path.split("/",3)[2]
  image_name = gs_path.split("/",3)[3]
  file_blob = getFileFromBucket(bucket_name, image_name)
  return file_blob

schema_instructions_text_FILE_URL = "gs://uk-bh-experiments-argolis-us/breedr/instructions.txt"  # @param {type: "string"}
# Read the contents of the file
instructions_file = getFileFromGSPath(schema_instructions_text_FILE_URL)
schema_instructions_text = instructions_file.download_as_text()
print(schema_instructions_text[:256])


In [ ]:
schema_instructions_text = """
dob: Is a field of date of birth in YYYY-MM-DD (ISO 8601) format in the attached animals table
date_moved_to_farm: Is a field of date that the animal was moved to the farm in YYYY-MM-DD (ISO 8601) format in the attached animals table
date_left_farm: Is a field of date that the animal left the farm in YYYY-MM-DD (ISO 8601) format in the attached animals table.
If this date_left_farm field is empty or NULL it means that the animal is still on the farm.
is_birthed: Is a boolean (TRUE or FALSE) field of date in the attached animals table that indicates whether an animal has given birth
is_on_farm: Is a boolean (TRUE or FALSE) field of date in the attached animals table that indicates whether the animal is still on the farm.
This should mean that the date_moved_to_farm is before the present date and the date_left_farm is NULL or empty.

Use the following logic to determine the gender of an animal based on the animals table:
IF is_male IS FALSE AND is_birthed IS TRUE THEN the gender is Cow
IF is_male IS FALSE AND is_birthed IS FALSE THEN the gender is Heifer
IF is_male IS TRUE AND is_castrated IS FALSE THEN the gender is Bull
IF is_male IS TRUE AND is_castrated IS TRUE THEN the gender is Steer

An animal is identified by the passport_number field.
If asked about a specific question about an animal such as, 'what is the date of birth for UK12345678?' return the information for that animal's row.
For weight information about animals, join the animal from animals table to  weights table using the field id in animals table to join to animal_id in  weights table.
he id field is unique in animals table but can be absent or have 1 or more corresponding rows in  weights table.
The weight field in  weights table is weight_value which gives an animal's weight in kilograms.
The current_date in  weights table indicates the date that the weight was recorded for the animal.
The age field in  weights table is the animal's age when the weight in weight_value was recorded
The field dlwg is the Daily Live Weight Gain for an animal which is the average amount of weight gained or lost per day by the animal since the previous weight.
If an animal does not have a previous weight, this field will be NULL or empty.
Report an animal's information using it's passport number not the id
Even if there is not weight information for all animals, report the information for the animals that do have weight information
 weights table can have the same animal on multiple rows, so group an animal's information before reporting the results when asked questions at the animal level.
 When reporting these results back use the passport number

Animal Breeds
To get an animal's breed do the following:
Left join the animals_animal_breeds from the animals_animal_breeds table to the animals table, using
animal.id = animals_animal_breeds.animal_id
Inner join the animal_breeds table from animal_breeds table to animals_animal_breeds,
using animals_animal_breeds.animalbreed_id = animal_breeds.id
Then report back animal_breeds.name as the breed name for an animal

Activities from activities table are joined to the animals using the animal table id field to left join to the activities table animal_id.

Fields in activities:
id: primary, unique key, do not report back, only used for internal joins
created_at: timestamp of when the activity was originally created
updated_at: timestamp of when the activity was last changed
activity_template_id: foreign key, inner joins to activities.activity_template_id = activity_template.id
created_by_id: id of the user who created the activity, do not report back
payload: A JSON payload. The JSON will need to be parsed to extract information in a clear format, using the following instructions:
Name of the activity: extracted from the top-level 'name'
Activity information: extracted from "fields". Depending on the activity, fields will contain a varying number of blocks within it.
Each block has its own unique title and from within each block the key pieces of information are: name, units, and value.
The value is the piece of data to report back, name is its title, and the units explains more of what the data is.
Convert the data to the appropriate field type, which can be a string, a numeric value, or a date. The fieldSlug is referenced by the long_description of activity_templates
business_unit_id: The business unit for the animal
animal_id: joins to the animal table, using animal.id = activities.animal_id
date: The date of when the activity occurred
activity_type_id: Foreign key, ignore for now
batch_id: Foreign key, ignore for now
fields: Foreign key, ignore for now
groups: Foreign key, ignore for now
unique_hash: Ignore for now
is_copied: Boolean, is this activity copied

The activities table joins to the activity_templates table in activity_templates table using activities.activity_template_id = activity_template.id

activity_template describes the data that is included in activities.
When information is queried for an animal or animals, the join will go from animals.id = activities.animal_id and then activities.template_activity_id = activity_templates.id.
The information in the activity.payload, as describe by activity_template.long_description, will be used to compare animals based on the supplied question.
Other fields will also be relevant, such as activity.date to give temporal information.

Fields in activity_templates:
id: Primary key, joins to activities.template_activity_id
name: Name of the activity
long_description: Provides a text string of the activity.
The string is a template which can be returned, with the items in curly brackets {} being references to the activity payload. These reference the values in the blocks, joined by fieldSlug which reports back the corresponding value in the block.
activity_type_id: Foreign key, ignore for now
business_unit_id: The business unit for the animal
created_by_id: id of the user who created the activity, do not report back
is_public: Boolean, is the activity public. If false, do not report back.
slug: String slug of the activity
is_disabled: Boolean, if TRUE ignore the activity and do not use or report its information
is_system: Boolean, ignore for now
handlers: Ignore
value_description: Ignore
short_description: Ignore
Is_allowed_to_delete: Ignore
allow_repeat_activity: Ignore
is_batchable: Ignore
"""

##  Agent: Convert the instructions to YAML

In [ ]:

schema_instructions_yaml = ask_data.convert_instructions_to_yaml(schema_instructions_text)
schema_instructions_yaml

# Answer User Questions

## Generate and format SQL to answer the question

### Agent: Generate SQL from the Question

In [ ]:
def generate_question_sql(question, schema_instructions_yaml, schema_json):

  generation_config = {
      "max_output_tokens": 8192,
      "temperature": 1,
      "top_p": 0.75,
  }
  schema_instructions_part = Part.from_text(text=schema_instructions_yaml)
  schema_part = Part.from_text(text=schema_json)

  prompt_part = Part.from_text(text= f"""
  <instructions>
    generate SQL to answer the question.

    Only generate a single SQL query, do not generate any other text.
    Use the schema in JSON format to ensure that you generate valid SQL.
    Generate valid BigQuery SQL using only legal BigQuery SQL syntax and BigQuery SQL functions.
    Use the SQL LIMIT statement to limit the number of rows returned so that the answer can be easily.
    Use SAFE_CAST to convert data types.
    For each column in the result make sure it has a meaningful name.
    Use the following schema given below.
    Only use columns that are in the schema.
    Use the following schema_instructions to understand the meaning of the tables and colunms, and the relationship between the tables.

    <question>
      {question}
    </question>

    </instructions>

    <examples>
      <example>
      <question>
      </question>
      <response>
        SELECT
          a.id,
          a.passport_number,
          a.dob,
          a.date_moved_to_farm,
          SAFE_CAST(a.is_male AS STRING) AS gender,
          SAFE_CAST(a.is_birthed AS STRING) AS is_birthed,
          SAFE_CAST(a.is_on_farm AS STRING) AS is_on_farm,
          ab.name AS breed_name,
          w.weight_value,
          w.current_date,
          w.age,
          w.dlwg
        FROM
          `uk-bh-experiments-argolis.breedr.animals` AS a
          LEFT OUTER JOIN `uk-bh-experiments-argolis.breedr.animals_animal_breeds` AS aab ON a.id = aab.animal_id
          LEFT OUTER JOIN `uk-bh-experiments-argolis.breedr.animal_breeds` AS ab ON aab.animalbreed_id = ab.id
          LEFT OUTER JOIN `uk-bh-experiments-argolis.breedr.weights` AS w ON a.id = w.animal_id
        WHERE ab.name = 'Hereford'
        ORDER BY
          w.weight_value DESC
        LIMIT 10
      </response>
    </example>

    <example>
      <question>
      </question>
      <response>
        SELECT
          f.name AS Field_Name,
          t.name AS Animal_Type,
          count(a.id) AS Animal_Count
        FROM
          `uk-bh-experiments-argolis.breedr.fields` AS f
          LEFT OUTER JOIN `uk-bh-experiments-argolis.breedr.animals` AS a ON f.id = a.field_id
          INNER JOIN `uk-bh-experiments-argolis.breedr.animal_types` AS t ON t.id  = a.animal_type_id

        GROUP BY 1, 2
        ORDER BY
          Field_Name ASC
      </response>
    </example>
  </examples>
  """
  )
  parts = prompt_part, schema_instructions_part, schema_part

  response = generate(parts=parts)
  raw_sql = response.text
  raw_sql = strip_sql_markdown(raw_sql)
  return raw_sql



### Agent: Add the project and dataset to the SQL if it's missing

In [ ]:
def add_dataset_to_sql(sql, project_id=PROJECT_ID, dataset_name=DATASET_NAME) -> str:
  generation_config = {
    "max_output_tokens": 8192,
    "temperature": 0.5,
    "top_p": 0.50,
  }
  sql_part = Part.from_text(text=sql)

  prompt_part = Part.from_text(text = f"""
  Add the following project and dataset to the SQL: {project_id}.{dataset_name}
  Use this to correct each table name, so that includes the dataset name.
  Output the SQL with no other changes.
  Only output the SQL with no other text.
  Make sure the SQL is valid BigQuery SQL.

  Here is an example:
  project_id:
    uk-bh-experiments-argolis
  dataset_name:
    breedr"
  sql:
    SELECT
        t2.dob as dob
      FROM
        `weights` AS t1
        INNER JOIN `animals` AS t2 ON t1.animal_id = t2.id
      WHERE t2.passport_number = 'UK286760601653'


  result:
    SELECT
        t2.dob as dob
      FROM
        `uk-bh-experiments-argolis.breedr.weights` AS t1
        INNER JOIN `uk-bh-experiments-argolis.breedr.animals` AS t2 ON t1.animal_id = t2.id
      WHERE t2.passport_number = 'UK286760601653'
  """
  )

  parts = [prompt_part, sql_part]
  response = generate(parts=parts)
      
  sql_with_dataset = response.text
  sql_with_dataset = strip_sql_markdown(sql_with_dataset)
  return sql_with_dataset

def format_sql(raw_sql, dataset_name="uk-bh-experiments-argolis.breedr.animals"):
  formatted_sql = add_dataset_to_sql(raw_sql, dataset_name)
  return formatted_sql

### Agent: Fix a SQL Error

In [ ]:
def fix_sql_error(sql, e: Exception, schema_instructions_yaml, schema_json) -> str:

  generation_config = {
      "max_output_tokens": 8192,
      "temperature": 2,
      "top_p": 0.99,
  }
  schema_instructions_part = Part.from_text(text=schema_instructions_yaml)
  schema_part = Part.from_text(text=schema_json)
  print(f"Fixing SQL error: {e}")

  prompt_part = Part.from_text(text= f"""
    You are a BigQuery SQL expert.
    The following SQL statement produces an exception.
    Produce a new version of this SQL that preserves it's intention and fixes the error.
    Make sure that the SQL you generate is different in some way to the original SQL.
    Orignal SQL:
      {sql}
    Exeption:
      {e}
    Only generate a single SQL query, do not generate any other text.
    Generate valid BigQuery SQL using only legal BigQuery SQL syntax and BigQuery SQL functions.
    Use the SQL LIMIT statement to limit the number of rows returned so that the answer can be easily.
    Use SAFE_CAST to convert data types when required.
    Use the provided schema  to ensure that you generate valid SQL: {schema_json}
    Only use columns that are in the schema.
    Pay attention to the data types of the columns that are provided in the schema.
    Use the following instructions to understand the meaning of the tables and columns: {schema_instructions_yaml}
  """
  )

  parts = [prompt_part, schema_instructions_part, schema_part]
  response = generate(parts=parts)
  raw_sql = response.text
  stripped_sql = strip_sql_markdown(raw_sql)
  return stripped_sql

## Run the Generated SQL

In [ ]:
from google.cloud import bigquery
DEBUG_TIMES =int(SQL_RETRY_TIMES)

def run_sql(sql, schema_instructions_yaml="", schema_json="",debug_times = 0):
  """

  Args:
    sql:the broken SQL
    schema_instructions_yaml, schema_json: instructions and schema for the SQL
    debug_times:  how many time shall we call this recursive function?

  Returns: fixed SQL

  """
  bq_client = bigquery.Client()
  answer = ""
  try:
    answer = bq_client.query(sql).to_dataframe()
    #print(f"Answer: {answer}")
  except Exception as e:
      if debug_times > 0:
        print(f"Generated SQL produced an error.  Fixing attempts: {debug_times}")
        fixed_sql = fix_sql_error(sql, e, schema_instructions_yaml, schema_json)
        print(f"Fixed SQL: {fixed_sql}")
        answer = run_sql(fixed_sql, schema_instructions_yaml, schema_json, debug_times - 1)
      else:
        print(f"Giving up. SQL {sql} produced error: {e}")
  return answer

def answer_question(question_text: str, schema_instructions_yaml: str, schema_json) -> str:
  question_sql = format_sql(generate_question_sql(question_text, schema_instructions_yaml, schema_json))
  print(f"Question SQL: {question_sql}")
  answer = run_sql(question_sql, schema_instructions_yaml, schema_json, DEBUG_TIMES)

  #answer = answer.to_markdown()
  return answer

## Agent: Explain the Results

In [ ]:
def explain_the_answer(question, answer, schema_instructions_yaml, schema_json):

  generation_config = {
      "max_output_tokens": 8192,
      "temperature": 1,
      "top_p": 0.95,
  }
  schema_instructions_part = Part.from_text(text=schema_instructions_yaml)
  schema_part = Part.from_text(text=schema_json)

  prompt_part = Part.from_text(text= f"""
    In 2-6 sentences, explain why the answer to the question: {question} is: {answer}
    Only generate a single explanation, do not generate any other text.
    Keep the explanation short and concise.
    To understand the database tables, use the following schema: {schema_json}
    To understand the database tables, use the following instructions to understand the meaning of the tables and columns: {schema_instructions_yaml}
  """
  )
  parts = [prompt_part,schema_part, schema_instructions_part]
  response = generate(parts=parts)
  text_explanation = response.text
  return text_explanation


In [ ]:
def explained_answer(question, schema_instructions_yaml=schema_instructions_yaml, schema_json=schema_json):
  answer = answer_question(question, schema_instructions_yaml, schema_json)
  #print(answer)

  explanation = explain_the_answer(question, answer, schema_instructions_yaml, schema_json)
  #print(f"Explanation: {explanation}")
  return answer, explanation

#Sample Questions

In [ ]:
answer,explanation = explained_answer("What is the date of birth for passport UK286760601653?")
answer
explanation

In [ ]:
answer,explanation = explained_answer( "Provide insights on the best-performing cattle based upon data provided.")
answer
explanation

In [ ]:
answer, explanation = explained_answer( "Which are my best cows? Analyse the data to find which cattle have the highest weights, highest rates of growth, and have been on the farm for the longest time. ")
answer
explanation

In [ ]:
answer, explanation = explained_answer( "What can you tell me about  UK323773305683?  How would the value of UK323773305683 compare to other animals?")
answer
explanation

In [ ]:
answer, explanation = explained_answer( "How would the value of UK323773305683 compare to other similar animals?")
answer
explanation

In [ ]:
answer, explanation = explained_answer( "How many of each type of animal are there in each field ?")
answer
explanation

In [ ]:
answer, explanation = explained_answer( "what is the total age and number cows who have given birth  in each field ?")
answer
explanation